# Post-Run Analysis: High-Level Performance Triage

This notebook analyzes the output of the `nsclc_eval` pipeline for each orchestrator–judge pair, compares judge consistency, and ranks orchestrators across the full all-to-all evaluation matrix.

> The results analyzed here are from the **anonymized 2026-07-16 evaluation session** stored in `results/session_2026-07-16/`. Patient identifiers have been replaced with sequential codes (P01, P02, ...) and file paths redacted, so no real patient data is present.

### What's Inside

1. **Data Collection** — recursively loads all `*_overall_summary.json` files into a flat Pandas DataFrame.
2. **The Matrix View** — groups by `[orchestrator_model, judge_model]` and shows RES + accuracy per pair.
3. **Judge Consistency** — Seaborn bar charts showing how each judge scored the same orchestrators.
4. **RES Distributions** — boxplot + stripplot overlays per orchestrator and per judge.
5. **Orchestrator Ranking** — a ranked leaderboard from the perspective of each judge.
6. **Confusion Matrices** — per judge-orchestrator pair, with Precision/Recall/F1 and ROC curves.
7. **Consensus Aggregation** — the final objective baseline metrics per orchestrator.

## 1. Data Collection & Consolidation

In [ ]:
import json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# === SET THE SESSION PATH HERE ===
EVAL_SESSION_PATH = "../results/session_2026-07-16"

def load_evaluation_data(session_path):
    records = []
    session_dir = Path(session_path)
    summary_files = list(session_dir.rglob('*overall_summary.json'))

    for file_path in summary_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            judge = data.get('judge_model', 'Unknown')

            for res in data.get('results', []):
                records.append({
                    'patient_id': res.get('patient_id'),
                    'orchestrator_model': res.get('orchestrator_model'),
                    'judge_model': judge,
                    'res_score': res.get('res_score', 0),
                    'binary_accuracy': res.get('binary_accuracy', 0),
                    'factuality_score': res.get('factuality_score', 0),
                    'faithfulness_score': res.get('faithfulness_score', 0),
                    'reasoning_completeness_score': res.get('reasoning_completeness_score', 0),
                    'treatment_completeness_score': res.get('treatment_completeness_score', 0),
                    'ground_truth_label': res.get('ground_truth_label'),
                    'predicted_treatment_label': res.get('predicted_label'),
                })
    return pd.DataFrame(records)

df = load_evaluation_data(EVAL_SESSION_PATH)
print(f'Loaded {len(df)} records.')
if not df.empty:
    display(df.sample(min(5, len(df))))
else:
    print('Warning: No records found. Please check EVAL_SESSION_PATH.')

## 2. Orchestrator–Judge Pair Evaluation Matrix

In [ ]:
pair_stats = df.groupby(['orchestrator_model', 'judge_model']).agg({
    'patient_id': ['count'],
    'res_score': ['mean', 'std'],
    'binary_accuracy': ['mean', 'std'],
    'factuality_score': ['mean', 'std'],
    'faithfulness_score': ['mean', 'std'],
    'reasoning_completeness_score': ['mean', 'std'],
    'treatment_completeness_score': ['mean', 'std'],
}).reset_index()

# Flatten MultiIndex columns
pair_stats.columns = ['_'.join(col).strip('_') for col in pair_stats.columns.values]
pair_stats = pair_stats.rename(columns={'patient_id_count': 'record_count'})

display(pair_stats)

## 3. Judge Consistency Analysis (isolating the orchestrator)

In [ ]:
# Short display labels
LABELS = {
    'baichuan-m2-32b':         'Baichuan',
    'gpt-oss-20b':             'GPT-OSS',
    'nemotron-3-nano-30B-A3B': 'Nemotron',
    'qwen3-30B-A3B-Thinking':  'Qwen3',
    'huatuoGPT-3-32b':         'HuatuoGPT',
}

metrics_to_plot = [
    ('res_score', 'Reasoning Efficiency Score (RES)'),
    ('binary_accuracy', 'Binary Accuracy'),
    ('factuality_score', 'Factuality Score'),
    ('faithfulness_score', 'Faithfulness Score'),
    ('reasoning_completeness_score', 'Reasoning Completeness Score (RCS)'),
    ('treatment_completeness_score', 'Treatment Completeness Score (TCS)'),
]

df_plot = df.copy()
df_plot['orchestrator_model'] = df_plot['orchestrator_model'].replace(LABELS)
df_plot['judge_model'] = df_plot['judge_model'].replace(LABELS)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.1)
x_order = sorted(df_plot['judge_model'].unique())
hue_order = sorted(df_plot['orchestrator_model'].unique())

for metric_col, metric_title in metrics_to_plot:
    plt.figure(figsize=(18, 7))
    ax = sns.barplot(
        data=df_plot, x='judge_model', y=metric_col, hue='orchestrator_model',
        order=x_order, hue_order=hue_order, errorbar='sd', capsize=0.05,
        edgecolor='.2', linewidth=1, width=.75,
    )
    plt.title(f'Performance Analysis: Average {metric_title} per Judge', pad=20, fontweight='bold', fontsize=18)
    plt.xlabel('Judge Model', fontweight='bold', fontsize=16)
    plt.ylabel(metric_title, fontweight='bold', fontsize=16)
    plt.xticks(ha='right', fontsize=14)
    plt.legend(title='Orchestrator Model', bbox_to_anchor=(1.01, 1), loc='upper left', title_fontsize=14, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Same metrics, grouped by orchestrator (isolating the judge)
x_order = sorted(df_plot['orchestrator_model'].unique())
hue_order = sorted(df_plot['judge_model'].unique())

for metric_col, metric_title in metrics_to_plot:
    plt.figure(figsize=(20, 8))
    ax = sns.barplot(
        data=df_plot, x='orchestrator_model', y=metric_col, hue='judge_model',
        order=x_order, hue_order=hue_order, errorbar='sd', capsize=0.05,
        edgecolor='.2', linewidth=1, width=.75,
    )
    plt.title(f'Performance Analysis: Average {metric_title} per Orchestrator', pad=20, fontweight='bold', fontsize=18)
    plt.xlabel('Orchestrator Model', fontweight='bold', fontsize=16)
    plt.ylabel(metric_title, fontweight='bold', fontsize=16)
    plt.xticks(ha='right', fontsize=16)
    plt.legend(title='Judge Model', bbox_to_anchor=(1.01, 1), loc='upper left', title_fontsize=16, fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. RES Distributions (boxplot + stripplot)

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
target_metric = 'res_score'
display_metric = target_metric.replace('_', ' ').title().replace('Res', 'RES')

orchestrators = sorted(df['orchestrator_model'].unique())
judges = sorted(df['judge_model'].unique())
n_orch = len(orchestrators)

fig = plt.figure(figsize=(18, 11))
if n_orch == 5:
    gs = gridspec.GridSpec(2, 6, figure=fig, hspace=0.6, wspace=0.3)
    axes = [
        fig.add_subplot(gs[0, 0:2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6]),
        fig.add_subplot(gs[1, 1:3]), fig.add_subplot(gs[1, 3:5]),
    ]
else:
    nrows = (n_orch + 2) // 3
    gs = gridspec.GridSpec(nrows, 3, figure=fig, hspace=0.6, wspace=0.3)
    axes = [fig.add_subplot(gs[r, c]) for r in range(nrows) for c in range(3)][:n_orch]

for i, orch in enumerate(orchestrators):
    ax = axes[i]
    orch_data = df[df['orchestrator_model'] == orch]
    sns.boxplot(
        data=orch_data, x='judge_model', y=target_metric, hue='judge_model',
        order=judges, hue_order=judges, palette=PALETTE, width=0.5,
        boxprops=dict(alpha=0.65, edgecolor='black', linewidth=1.5),
        medianprops=dict(color='black', linewidth=2.5),
        whiskerprops=dict(color='black', linewidth=1.5),
        capprops=dict(color='black', linewidth=1.5),
        showfliers=False, legend=False, ax=ax,
    )
    sns.stripplot(
        data=orch_data, x='judge_model', y=target_metric, hue='judge_model',
        order=judges, hue_order=judges, palette=PALETTE, size=5.5, alpha=0.75,
        jitter=0.15, ax=ax, legend=False,
    )
    ax.set_title(f'Orchestrator: {orch}', pad=10, fontweight='bold', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(display_metric, fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    y_bottom, y_top = ax.get_ylim()
    ax.set_ylim(y_bottom, y_top + (y_top - y_bottom) * 0.08)

legend_handles = [
    mpatches.Patch(facecolor=color, edgecolor='black', alpha=0.7, label=judge)
    for judge, color in zip(judges, PALETTE[:len(judges)])
]
fig.legend(handles=legend_handles, title='Judge Model', title_fontsize=12, fontsize=10.5,
           loc='lower center', bbox_to_anchor=(0.5, -0.07), ncol=min(len(judges), 5),
           frameon=True, facecolor='white', edgecolor='#CCCCCC', framealpha=0.95)
fig.suptitle(f'{display_metric} Distribution across Judge Models', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0.08, 1, 0.96])
plt.show()

## 5. Orchestrator Ranking (isolating the judge)

In [ ]:
# Rank orchestrators by average RES score for each judge
ranking = df.groupby(['judge_model', 'orchestrator_model'])['res_score'].mean().reset_index()
ranking = ranking.sort_values(['judge_model', 'res_score'], ascending=[True, False])

for judge in ranking['judge_model'].unique():
    print(f'\n--- Rankings according to {judge} ---')
    judge_ranking = ranking[ranking['judge_model'] == judge]
    for i, row in enumerate(judge_ranking.itertuples(), 1):
        print(f"{i}. {row.orchestrator_model} (RES: {row.res_score:.2f})")

## 6. Confusion Matrices by Judge and Orchestrator

In [ ]:
from sklearn.metrics import confusion_matrix

orchestrator_models = sorted(df['orchestrator_model'].unique())
judge_models = sorted(df['judge_model'].unique())
num_orchestrators = len(orchestrator_models)
num_judges = len(judge_models)

fig, axes = plt.subplots(num_judges, num_orchestrators,
                         figsize=(num_orchestrators * 4.5, num_judges * 3.5), dpi=200)

for i, judge in enumerate(judge_models):
    for j, orch in enumerate(orchestrator_models):
        ax = axes[i, j]
        subset = df[(df['judge_model'] == judge) & (df['orchestrator_model'] == orch)]
        if not subset.empty:
            subset = subset.dropna(subset=['ground_truth_label', 'predicted_treatment_label'])
            if not subset.empty:
                y_true = subset['ground_truth_label'].astype(int)
                y_pred = subset['predicted_treatment_label'].astype(int)
                cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                            annot_kws={'size': 20, 'weight': 'bold'},
                            xticklabels=['IO [0]', 'IOCT [1]'], yticklabels=['IO [0]', 'IOCT [1]'])
                ax.tick_params(axis='both', which='major', labelsize=14)
                if i == 0:
                    ax.set_title(f'Orch: {orch}', fontsize=18, pad=15)
                if j == 0:
                    ax.set_ylabel(f'Judge: {judge}\n\nActual', fontsize=16, rotation=90, labelpad=15)
                else:
                    ax.set_ylabel('')
                    ax.set_yticks([])
                if i == num_judges - 1:
                    ax.set_xlabel('Predicted', fontsize=16, labelpad=10)
                else:
                    ax.set_xlabel('')
                    ax.set_xticks([])
            else:
                ax.axis('off')
        else:
            ax.axis('off')
            ax.set_title(f'Orch: {orch}\nJudge: {judge}\n(No Data)', fontsize=16, color='gray')

plt.tight_layout()
plt.suptitle('Confusion Matrices by Judge and Orchestrator', fontsize=22, fontweight='bold', y=1.04)
plt.show()

### Interpreting the confusion matrices

Reading a single 2×2 matrix (IOCT = class **1**, IO = class **0**):

* **True Negative (top-left)** — correctly predicted IO.
* **False Positive (top-right)** — incorrectly predicted IOCT for an IO patient.
* **False Negative (bottom-left)** — missed an IOCT patient.
* **True Positive (bottom-right)** — correctly predicted IOCT.

From these four counts one can derive **recall/sensitivity**, **precision**, **accuracy** and **F1**.

**Typical findings on the 2026-07-16 toy cohort:**

* A **systemic positive bias** toward predicting `IOCT [1]` (high sensitivity, frequent false positives).
* **Judge rigidity** — advanced judges such as Nemotron and Qwen3-Thinking produce nearly identical matrices regardless of the orchestrator.
* **Self-evaluation** — Baichuan-M2-32B is the only model whose self-evaluation is notably balanced (near-zero false positives).

## 7. Precision, Recall and F1 by Judge

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

df_plot = df.copy()
df_plot['orchestrator_model'] = df_plot['orchestrator_model'].replace(LABELS)
df_plot['judge_model'] = df_plot['judge_model'].replace(LABELS)

orchestrator_models = sorted(df_plot['orchestrator_model'].unique())
judge_models = sorted(df_plot['judge_model'].unique())

fig, axes = plt.subplots(1, len(judge_models), figsize=(28, 7), dpi=200, sharey=True)
width = 0.25

for i, judge in enumerate(judge_models):
    ax = axes[i]
    precisions, recalls, f1s, valid_orchs = [], [], [], []
    for orch in orchestrator_models:
        subset = df_plot[(df_plot['judge_model'] == judge) & (df_plot['orchestrator_model'] == orch)]
        subset = subset.dropna(subset=['ground_truth_label', 'predicted_treatment_label'])
        if not subset.empty:
            y_true = subset['ground_truth_label'].astype(int)
            y_pred = subset['predicted_treatment_label'].astype(int)
            precisions.append(precision_score(y_true, y_pred, zero_division=0))
            recalls.append(recall_score(y_true, y_pred, zero_division=0))
            f1s.append(f1_score(y_true, y_pred, zero_division=0))
            valid_orchs.append(orch)
    x = np.arange(len(valid_orchs))
    b1 = ax.bar(x - width, precisions, width, label='Precision', color='#4c72b0')
    b2 = ax.bar(x, recalls, width, label='Recall', color='#dd8452')
    b3 = ax.bar(x + width, f1s, width, label='F1 Score', color='#55a868')
    ax.bar_label(b1, fmt='%.2f', fontsize=9, padding=4, rotation=90)
    ax.bar_label(b2, fmt='%.2f', fontsize=9, padding=4, rotation=90)
    ax.bar_label(b3, fmt='%.2f', fontsize=9, padding=4, rotation=90)
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    if i == 0:
        ax.set_ylabel('Score (0.0 to 1.0)', fontsize=16, fontweight='bold')
    ax.set_title(f'Judge: {judge}', fontsize=16, fontweight='bold', pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(valid_orchs, rotation=45, ha='right', fontsize=14)
    ax.set_ylim([0, 1.15])
    if i == 0:
        ax.legend(loc='upper right', fontsize=12)

plt.tight_layout()
plt.suptitle('Precision, Recall, and F1 Score by Judge Model', fontsize=22, fontweight='bold', y=1.05)
plt.show()

## 8. ROC Curves & AUC

In [ ]:
from sklearn.metrics import roc_curve, auc

orchestrator_models = sorted(df_plot['orchestrator_model'].unique())
judge_models = sorted(df_plot['judge_model'].unique())

fig = plt.figure(figsize=(20, 13), dpi=200)
gs = gridspec.GridSpec(2, 6, figure=fig, hspace=0.4, wspace=0.8)
axes = [
    fig.add_subplot(gs[0, 1:3]), fig.add_subplot(gs[0, 3:5]),
    fig.add_subplot(gs[1, 0:2]), fig.add_subplot(gs[1, 2:4]), fig.add_subplot(gs[1, 4:6]),
]

for i, judge in enumerate(judge_models):
    ax = axes[i]
    for orch in orchestrator_models:
        subset = df_plot[(df_plot['judge_model'] == judge) & (df_plot['orchestrator_model'] == orch)]
        subset = subset.dropna(subset=['ground_truth_label', 'predicted_treatment_label'])
        if not subset.empty:
            y_true = subset['ground_truth_label'].astype(int)
            y_score = subset['predicted_treatment_label'].astype(int)
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, lw=2.5, label=f'{orch} (AUC={roc_auc:.2f})')
    ax.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
    ax.set_xlim([-0.05, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=16, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=16, fontweight='bold')
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_title(f'Judge:\n{judge}', fontsize=18, fontweight='bold', pad=10)
    ax.legend(title='Orchestrator Model', title_fontsize=13, loc='lower right', fontsize=11.5)

plt.suptitle('ROC Curves & AUC by Judge Model', fontsize=24, fontweight='bold', y=0.97)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 9. RES Crash Detection (zero-score cases)

In [ ]:
# Find cases where the RES score crashed to exactly 0
failed_cases = df[df['res_score'] == 0]
print(f'Found {len(failed_cases)} failed case(s) with RES = 0.')
display(failed_cases[['patient_id', 'orchestrator_model', 'judge_model', 'res_score']])

# Optionally drop them for downstream analysis:
# df = df[df['res_score'] > 0].copy()

## 10. Radar Chart (6-metric overview per judge)

In [ ]:
from math import pi

PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
metrics = ['res_score', 'binary_accuracy', 'factuality_score', 'faithfulness_score',
           'reasoning_completeness_score', 'treatment_completeness_score']
metric_labels = ['RES', 'Accuracy', 'Factuality', 'Faithfulness', 'Reasoning\nCompleteness', 'Treatment\nCompleteness']

spider_df = df.groupby(['judge_model', 'orchestrator_model'])[metrics].mean().reset_index()
spider_df['binary_accuracy'] = spider_df['binary_accuracy'] * 100
spider_df['faithfulness_score'] = ((spider_df['faithfulness_score'] - 1) / 4) * 100

judges = sorted(spider_df['judge_model'].unique())
orchestrators = sorted(spider_df['orchestrator_model'].unique())
colors = PALETTE[:len(orchestrators)]

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10), subplot_kw=dict(polar=True))
axes = axes.flatten()
N = len(metrics)
angles = [n / float(N) * 2 * pi for n in range(N)] + [0.0]

for i, judge in enumerate(judges):
    ax = axes[i]
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_labels, fontsize=10, fontweight='bold', color='black')
    ax.set_rlabel_position(30)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(['20', '40', '60', '80', '100'], color='grey', size=8)
    ax.set_ylim(0, 100)
    ax.set_title(f'Judge: {judge}', size=13, fontweight='bold', position=(0.5, 1.15))
    judge_data = spider_df[spider_df['judge_model'] == judge]
    for j, orch in enumerate(orchestrators):
        orch_data = judge_data[judge_data['orchestrator_model'] == orch]
        if orch_data.empty:
            continue
        values = orch_data[metrics].values.flatten().tolist() + [0.0]
        values[-1] = values[0]
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=orch, color=colors[j])
        ax.fill(angles, values, color=colors[j], alpha=0.1)

if len(judges) < len(axes):
    for j in range(len(judges), len(axes)):
        fig.delaxes(axes[j])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center', bbox_to_anchor=(0.83, 0.25),
           title='Orchestrator Model', fontsize=11, title_fontsize=13, frameon=True)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()

## 11. Consensus Aggregation (The Final Benchmark)

In [ ]:
# Consensus scores per patient-orchestrator pair.
# Continuous metrics: mean. Binary accuracy: majority vote.
consensus_df = df.groupby(['patient_id', 'orchestrator_model']).agg({
    'res_score': 'mean',
    'factuality_score': 'mean',
    'faithfulness_score': 'mean',
    'reasoning_completeness_score': 'mean',
    'treatment_completeness_score': 'mean',
    'binary_accuracy': lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0],
}).reset_index()

final_metrics = consensus_df.groupby('orchestrator_model').agg({
    'res_score': ['mean', 'std'],
    'binary_accuracy': 'mean',
    'factuality_score': 'mean',
    'faithfulness_score': 'mean',
}).reset_index()

final_metrics.columns = ['_'.join(col).strip('_') for col in final_metrics.columns.values]
final_metrics = final_metrics.sort_values('res_score_mean', ascending=False)

print('=== Final Consensus Baseline Metrics ===')
display(final_metrics)